<img src="../docs/banniere_projets_GM.png" width="70%" /><br>

-----
<img src="../docs/uber_logo.png" width="30%" /><br>

# Uber Pickups

>Ce projet est obligatoire pour le bloc 3 (Analyse prédictive de données structurées par l'intelligence artificielle) de la certification CDSD.\
>Concepteur développeur en science des données | RNCP35288

---

## 🎯 Objectif
Uber souhaite optimiser son process de prise en charge des clients en aidant ses chauffeurs à se positionner au bon endroit, au bon moment.
Quand un client attend plus de 5 à 7 minutes, il finit souvent par annuler sa course.
L'idée de ce projet est donc d'exploiter les données historiques de pickups (prises en charge) pour repérer les **zones chaudes** — les endroits de New York où la demande est la plus forte — et les recommander aux chauffeurs.

Le projet suit les différentes parties suggérées :

1. Setup, chargement des données et préparation des features temporelles
2. EDA (à quelles heures / quels jours la demande est-elle forte ?)
3. Clustering avec **KMeans** — on commence petit puis on généralise
4. Clustering avec **DBSCAN** — une approche basée sur la densité
5. Comparaison des deux approches et conclusion

## 📊 Source des données
On dispose des données de pickups de **2014, d'avril à septembre** (6 fichiers csv, ~4,5 millions de courses).<br> Disponibles dans le dossier `/data/raw`.

---
## 1 - Import des librairies et chargement des données

In [33]:
import glob
import os

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

On charge les 6 fichiers CSV (avril → septembre 2014). Chacun contient 4 colonnes :
`Date/Time`, `Lat`, `Lon` et `Base`. On les renomme en minuscules pour travailler plus facilement.

In [34]:
files = sorted(glob.glob('../data/raw/uber-raw-data-*14.csv'))
files

['../data/raw\\uber-raw-data-apr14.csv',
 '../data/raw\\uber-raw-data-aug14.csv',
 '../data/raw\\uber-raw-data-jul14.csv',
 '../data/raw\\uber-raw-data-jun14.csv',
 '../data/raw\\uber-raw-data-may14.csv',
 '../data/raw\\uber-raw-data-sep14.csv']

In [35]:
dfs = []
for f in files:
    tmp = pd.read_csv(f)
    tmp.columns = ['datetime', 'lat', 'lon', 'base']
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'Total pickups : {len(df):,}')
df.head()

Total pickups : 4,534,327


,datetime,lat,lon,base
0,4/1/2014 0:11:00,40.7690,-73.9549,B02512
1,4/1/2014 0:17:00,40.7267,-74.0345,B02512
2,4/1/2014 0:21:00,40.7316,-73.9873,B02512
3,4/1/2014 0:28:00,40.7588,-73.9776,B02512
4,4/1/2014 0:33:00,40.7594,-73.9722,B02512


In [61]:
df.dtypes

datetime     datetime64[ns]
lat                 float64
lon                 float64
base                 object
hour                  int32
dayofweek             int32
dayname              object
month                 int32
dtype: object

On convertit la colonne `datetime` en vrai format date, puis on en extrait les
informations qui vont nous servir : l'heure, le jour de la semaine et le mois.

In [36]:
df['datetime']  = pd.to_datetime(df['datetime'])

df['hour']      = df['datetime'].dt.hour
df['dayofweek'] = df['datetime'].dt.dayofweek   # 0 = lundi ... 6 = dimanche
df['dayname']   = df['datetime'].dt.day_name()
df['month']     = df['datetime'].dt.month

df.head()

,datetime,lat,lon,base,hour,dayofweek,dayname,month
0,2014-04-01 00:11:00,40.7690,-73.9549,B02512,0,1,Tuesday,4
1,2014-04-01 00:17:00,40.7267,-74.0345,B02512,0,1,Tuesday,4
2,2014-04-01 00:21:00,40.7316,-73.9873,B02512,0,1,Tuesday,4
3,2014-04-01 00:28:00,40.7588,-73.9776,B02512,0,1,Tuesday,4
4,2014-04-01 00:33:00,40.7594,-73.9722,B02512,0,1,Tuesday,4


On vérifie rapidement qu'il n'y a pas de valeurs manquantes.

In [37]:
df.isna().sum()

datetime     0
lat          0
lon          0
base         0
hour         0
dayofweek    0
dayname      0
month        0
dtype: int64

On prépare aussi deux listes pour ordonner et traduire les jours de la semaine que l'on réutilisera par la suite.

In [38]:
day_order_en = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_order_fr = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
day_fr = dict(zip(day_order_en, day_order_fr))

---
## 2 - Analyse exploratoire (EDA)

Avant de lancer le moindre algorithme, on regarde **quand** la demande est forte.
Cela nous permettra de focaliser sur un créneau pertinent pour démarrer le clustering.

### 2.1 - <u>Pickups par heure de la journée</u>

In [40]:
by_hour = df.groupby('hour').size().reset_index(name='pickups')

fig = px.bar(by_hour, x='hour', y='pickups',
             title='Nombre de pickups par heure de la journée',
             labels={'hour': 'Heure', 'pickups': 'N pickups'},
             color='pickups', color_continuous_scale='Reds')

fig.update_layout(
    title=dict(
        text='<b>Nombre de pickups par heure de la journée</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.show()

>><u>**Observations :**</u><br>
>>
>>On retrouve deux pics logiques :<br>
>>- un le matin (vers 7h-9h),
>>- et surtout un gros pic en fin d'après-midi / début de soirée (16h-19h), au moment où les gens rentrent ou sortent.

### 2.2 - <u>Pickups par jour de la semaine</u>

In [45]:
by_day = df.groupby('dayname').size().reset_index(name='pickups')
by_day['jour']  = by_day['dayname'].map(day_fr)
by_day['order'] = by_day['dayname'].map({d: i for i, d in enumerate(day_order_en)})
by_day = by_day.sort_values('order')

fig = px.bar(by_day, x='jour', y='pickups',
             title='Nombre de pickups par jour de la semaine',
             labels={'jour': 'Jour', 'pickups': 'N pickups'},
             color='pickups', color_continuous_scale='Reds')

fig.update_layout(
    title=dict(
        text='<b>Nombre de pickups par jour de la semaine</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.show()

>><u>**Observation</u> :**<br>
>>
>>La demande monte progressivement en semaine et culmine le **jeudi** et le **vendredi**.

### 2.3 - <u>Visualisation synthétique (jour x heure)</u>

En croisant le jour et l'heure, on visualise d'un seul coup d'œil via une heatmap les créneaux
les plus chargés.

In [50]:
heat = df.groupby(['dayofweek', 'hour']).size().reset_index(name='pickups')
heat_piv = heat.pivot(index='dayofweek', columns='hour', values='pickups')
heat_piv.index = day_order_fr

fig = px.imshow(heat_piv,
                title='Heatmap 2D des pickups par jour et par heure',
                labels=dict(x='Heure', y='Jour', color='Pickups'),
                color_continuous_scale='Reds', aspect='auto')

fig.update_layout(
    title=dict(
        text='<b>Heatmap 2D des pickups par jour et par heure</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.show()

>><u>**Conclusion sur l'EDA</u> :**
>>
>>- Le pic de demande est en fin d'après-midi / soirée (16h-19h).
>>- Le jeudi et le vendredi soir sont les moments les plus chargés.
>>- La nuit (1h-5h) est calme en semaine, mais reste active le week-end.

On va donc commencer le clustering sur l'un de ces pics : **le vendredi à 18h**.

---
## 3 - Clustering avec KMeans

L'idée du clustering : regrouper les pickups proches géographiquement en quelques
groupes (clusters). Le **centre** de chaque groupe est une zone chaude à recommander.

KMeans demande de fixer à l'avance le nombre de clusters `k`. On part sur `k = 8`
zones, ce qui est un bon compromis pour New York.

Pour préparer les cartes statiques, on écrit une petite fonction matplotlib qui
affiche les points (longitude en x, latitude en y) colorés par cluster.

In [51]:
def save_map(data, color_col, title, name, centers=None):
    # Carte statique (PNG) : longitude en x, latitude en y, couleur = cluster
    fig, ax = plt.subplots(figsize=(9, 9))
    cmap = plt.cm.tab10
    categories = sorted(data[color_col].unique(), key=lambda x: (len(str(x)), str(x)))

    for i, cat in enumerate(categories):
        sub = data[data[color_col] == cat]
        ax.scatter(sub['lon'], sub['lat'], s=4, alpha=0.4, color=cmap(i % 10), label=str(cat))

    # On peut superposer les centres de clusters (croix noires)
    if centers is not None:
        ax.scatter(centers['lon'], centers['lat'], s=200, c='black', marker='X',
                   edgecolors='white', linewidths=1.5, label='Centres', zorder=5)

    ax.set_xlim(-74.1, -73.7)
    ax.set_ylim(40.6, 40.9)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend(markerscale=3, fontsize=8, loc='upper left')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=130)
    plt.close(fig)

### 3.1 - <u>Commencer petit : le vendredi à 18h</u>

On isole les pickups du vendredi à 18h et on applique KMeans dessus.
C'est notre test sur un cas concret avant de généraliser.

In [74]:
fri_6pm = df[(df['dayofweek'] == 4) & (df['hour'] == 18)].copy()
print(f'Pickups vendredi 18h : {len(fri_6pm):,}')

km_small = KMeans(n_clusters=8, random_state=42, n_init=10)
fri_6pm['cluster'] = km_small.fit_predict(fri_6pm[['lat', 'lon']].values).astype(str)

# Échantillon pour alléger l'affichage de la carte
sample = fri_6pm.sample(min(5000, len(fri_6pm)), random_state=42)

fig = px.scatter_map(sample, lat='lat', lon='lon', color='cluster',
                     map_style='carto-positron', zoom=10,
                     center={'lat': 40.73, 'lon': -73.99},
                     title='KMeans (k=8) — Vendredi 18h', opacity=0.8)

fig.update_layout(
    title=dict(
        text='<b>KMeans (k=8) — Vendredi 18h</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.update_traces(marker={'size': 6})

fig.show()

Pickups vendredi 18h : 54,762


KMeans découpe bien la ville en 8 zones cohérentes, concentrées sur Manhattan
et ses alentours. L'approche fonctionne, on peut généraliser.

### 3.2 - <u>Généralisation : une carte des zones chaudes par jour</u>

On applique maintenant KMeans **sur chaque jour de la semaine** et on ne garde que
les **centres** des clusters. Ces centres sont directement les zones chaudes à
recommander aux chauffeurs ce jour-là.

In [72]:
centers_list = []
for i, day in enumerate(day_order_en):
    data = df[df['dayname'] == day][['lat', 'lon']].dropna()
    km = KMeans(n_clusters=8, random_state=42, n_init=10)
    km.fit(data)

    c = pd.DataFrame(km.cluster_centers_, columns=['lat', 'lon'])
    c['jour'] = day_order_fr[i]
    centers_list.append(c)

centers_df = pd.concat(centers_list, ignore_index=True)
centers_df.head()

,lat,lon,jour
0,40.731575,-73.998039,Lundi
1,40.771920,-73.532490,Lundi
2,40.687138,-73.964775,Lundi
3,40.766333,-73.972335,Lundi
4,40.886447,-73.892615,Lundi


In [82]:
fig = px.scatter_map(centers_df, lat='lat', lon='lon', color='jour',
                     map_style='carto-positron', zoom=9,
                     center={'lat': 40.73, 'lon': -73.99},
                     title='Zones chaudes (centres KMeans) par jour de la semaine',
                     opacity=0.8)

fig.update_layout(
    title=dict(
        text='<b>Zones chaudes (centres KMeans) par jour de la semaine</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.update_traces(marker={'size': 12})

fig.show()

>><u>**Observations</u> :**
>>
>>On remarque que les zones chaudes sont **très stables d'un jour à l'autre** :
les centres se superposent presque tous. Cela veut dire que les chauffeurs peuvent
se fier à un petit nombre de zones récurrentes (Midtown, Lower Manhattan, les
aéroports, les grandes gares), quel que soit le jour.

---
## 4 - Clustering avec DBSCAN

DBSCAN regroupe les points en fonction de leur **densité**.<br>
**Gros avantage :** pas besoin de fixer le nombre de clusters à l'avance, il le trouve tout seul.<br>
Les points trop isolés sont classés comme **bruit** (cluster = -1).

Deux paramètres comptent :
- `eps` (epsilon) : la distance maximale pour considérer deux points comme voisins ;
- `min_samples` : le nombre minimum de voisins pour former une zone dense.

On normalise les coordonnées avec un `StandardScaler` (DBSCAN est sensible aux échelles)
et on travaille sur un échantillon de 50 000 points pour garder un temps de calcul raisonnable.

In [77]:
df_db = df.sample(50000, random_state=42).copy()

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(df_db[['lat', 'lon']].values)

db = DBSCAN(eps=0.2, min_samples=100)
df_db['cluster'] = db.fit_predict(coords_scaled)

n_clusters = len(set(df_db['cluster'])) - (1 if -1 in df_db['cluster'].values else 0)
n_noise    = (df_db['cluster'] == -1).sum()
print(f'Clusters détectés : {n_clusters}')
print(f'Points classés en bruit : {n_noise:,} ({n_noise / len(df_db) * 100:.1f}%)')

Clusters détectés : 6
Points classés en bruit : 2,481 (5.0%)


In [83]:
# On retire le bruit pour la visualisation
df_db_clean = df_db[df_db['cluster'] != -1].copy()
df_db_clean['cluster'] = df_db_clean['cluster'].astype(str)

fig = px.scatter_map(df_db_clean, lat='lat', lon='lon', color='cluster',
                     map_style='carto-positron', zoom=10,
                     center={'lat': 40.73, 'lon': -73.99},
                     title='DBSCAN — zones denses', opacity=0.8)

fig.update_layout(
    title=dict(
        text='<b>DBSCAN — zones denses</b>',
        x=0.5,
        xanchor='center'
    )
)

fig.update_traces(marker={'size': 6})

fig.show()

>><u>**Observations</u> :**
>>
>>DBSCAN dégage un énorme cluster central qui épouse la forme de Manhattan, plus
quelques clusters isolés bien nets qui correspondent aux **aéroports** (JFK, LaGuardia)
et à **Newark**. C'est cohérent avec ce qu'on attendait : ce sont des points de forte
demande, géographiquement séparés du reste.

---
## 5 - Comparaison KMeans vs DBSCAN

In [86]:
comparison = pd.DataFrame({
    'Critère': ['Nombre de clusters', 'Points non assignés', 'Forme des clusters',
                'Paramètre clé', 'Vitesse'],
    'KMeans': ['Fixé à l\'avance (k=8)', 'Aucun (tous assignés)', 'Plutôt régulière',
               'Nombre de clusters k', 'Très rapide'],
    'DBSCAN': [f'Trouvé automatiquement ({n_clusters})', f'{n_noise:,} points de bruit',
               'Variable (suit la densité)', 'eps, min_samples', 'Plus lent sur gros volumes']
})
comparison

,Critère,KMeans,DBSCAN
0,Nombre de clusters,Fixé à l'avance (k=8),Trouvé automatiquement (6)
1,Points non assignés,Aucun (tous assignés),"2,481 points de bruit"
2,Forme des clusters,Plutôt régulière,Variable (suit la densité)
3,Paramètre clé,Nombre de clusters k,"eps, min_samples"
4,Vitesse,Très rapide,Plus lent sur gros volumes


---
## 6. Conclusion générale

Les deux algorithmes (KMeans vs. DBSCAN) donnent des résultats cohérents et complémentaires.

- **KMeans** est simple, rapide et facile à interpréter. En fixant 8 zones par jour,
  on obtient une carte claire des hot-zones à recommander. Les centres tombent
  logiquement sur Midtown, Lower Manhattan, les aéroports et les grandes gares,
  et restent stables d'un jour à l'autre.

- **DBSCAN** ne demande pas de fixer le nombre de clusters et révèle bien la
  structure de la demande : un grand bloc sur Manhattan et des poches isolées sur
  les aéroports. En contrepartie, il génère du bruit et ses paramètres demandent
  un peu de réglage.

**En pratique pour Uber :** KMeans est le plus adapté pour produire la
recommandation "voici les N zones où te placer aujourd'hui", tandis que DBSCAN
sert à valider que ces zones correspondent bien à des poches de demande réelles et permet également de détecter de nouvelles zones émergentes.<br><br>

---
---